# ⚡ Boosting
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> While Random Forests train trees independently and average them, Boosting trains trees *sequentially* — each tree learns from the mistakes of the previous one. This iterative error-correction makes Boosting one of the highest-performing ML approaches.

### What you'll learn
- AdaBoost — upweighting misclassified points
- Gradient Boosting — fitting trees to residuals
- XGBoost — the industry-standard implementation
- Key hyperparameters: n_estimators, learning_rate, max_depth
- Comparing Boosting vs Random Forests

### Dataset: California Housing (regression) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
!pip install xgboost -q
import xgboost as xgb
print("✓ XGBoost:", xgb.__version__)

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']
print(f"California Housing: {df_house.shape}")
df_house.head()

In [ ]:
# Load California Housing
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"California Housing: {X.shape[0]:,} census blocks")
print(f"Predicting: median house value (in $100,000s)")
print(f"Features: {list(X.columns)}")

## ⚡ Part 1 — How Gradient Boosting Works

**Core idea:** Train trees sequentially. Each tree fits the *residuals* (errors) from all previous trees.

**Step by step:**
1. Initialize with a simple prediction (e.g. mean of Y)
2. Compute residuals: rᵢ = yᵢ - ŷᵢ
3. Fit a small tree to the residuals
4. Update: ŷᵢ ← ŷᵢ + λ × (tree prediction)  (λ = learning rate, shrinks each step)
5. Repeat 2-4 for B iterations

The **learning rate** (shrinkage) λ is critical — small λ requires more trees but generalizes better.

In [ ]:
# Show boosting building up — predictions improve with each tree
from sklearn.ensemble import GradientBoostingRegressor

stages = [1, 5, 25, 100, 300]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

# Use a small 1D slice for visualization
X_viz = X_tr[['MedInc']].copy()
y_viz = y_tr.values

for ax, n in zip(axes, stages):
    gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1,
                                     max_depth=3, random_state=42)
    gbm.fit(X_viz, y_viz)
    
    xi = np.linspace(X_viz['MedInc'].min(), X_viz['MedInc'].max(), 200).reshape(-1,1)
    yi_pred = gbm.predict(xi)
    
    ax.scatter(X_viz['MedInc'], y_viz, alpha=0.1, s=5, color='#888')
    ax.plot(xi, yi_pred, color='#e85d20', lw=2)
    mse = mean_squared_error(y_viz, gbm.predict(X_viz))
    ax.set_title(f'n={n}\nRMSE={np.sqrt(mse):.3f}', fontsize=10)
    ax.set_xlabel('Median Income')
    if n==1: ax.set_ylabel('House Value')

plt.suptitle('Gradient Boosting — Predictions Improve with More Trees', y=1.05, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: sklearn GBM vs XGBoost
from sklearn.metrics import mean_squared_error as mse

models = {
    'GBM (sklearn)': GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                                max_depth=4, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, learning_rate=0.1,
                                  max_depth=4, random_state=42,
                                  eval_metric='rmse', verbosity=0),
}

print(f"{'Model':<20} {'Train RMSE':>12} {'Test RMSE':>12}")
print("-"*45)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    tr_rmse = np.sqrt(mse(y_tr, model.predict(X_tr)))
    te_rmse = np.sqrt(mse(y_te, model.predict(X_te)))
    print(f"{name:<20} {tr_rmse:>12.4f} {te_rmse:>12.4f}")
print("\n📌 XGBoost uses second-order gradients, regularization, and parallel computing")
print("   — often faster and slightly better than sklearn's GBM")

In [ ]:
# Effect of learning rate and n_estimators (the key tradeoff)
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3]
n_estimators   = [10, 50, 100, 200, 500]
results = np.zeros((len(learning_rates), len(n_estimators)))

for i, lr in enumerate(learning_rates):
    for j, n in enumerate(n_estimators):
        gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                         max_depth=3, random_state=42)
        gbm.fit(X_tr, y_tr)
        results[i,j] = np.sqrt(mse(y_te, gbm.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 4))
for i, lr in enumerate(learning_rates):
    ax.plot(n_estimators, results[i], 'o-', lw=1.8, markersize=5,
            label=f'lr={lr}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('Test RMSE')
ax.set_title('Learning Rate × n_estimators Tradeoff\n(small lr requires many trees but generalizes better)')
ax.legend(title='Learning rate', fontsize=9)
plt.tight_layout()
plt.show()
print("\n📌 Small learning rate (0.01) + many trees often outperforms large lr + few trees")
print("   But small lr is slower to train — XGBoost's early stopping solves this")

In [ ]:
# Feature importance (XGBoost)
xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
xgb_model.fit(X_tr, y_tr)

imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e85d20' if v > imp.quantile(0.6) else '#1e5fa8' for v in imp]
ax.barh(imp.index, imp.values, color=colors, edgecolor='white')
ax.set_title('XGBoost Feature Importance — California Housing')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()
final_rmse = np.sqrt(mse(y_te, xgb_model.predict(X_te)))
print(f"\nFinal XGBoost Test RMSE: {final_rmse:.4f}")
print("📌 MedInc (median income) dominates — wealthy neighborhoods = higher house values")

In [ ]:
answers = {
    "q1": "",  # What is the core difference between Random Forest and Gradient Boosting?
    "q2": "",  # What does the learning rate control in gradient boosting?
    "q3": "",  # Why does a very high learning rate (e.g. 0.9) often perform poorly?
    "q4": "",  # What is the advantage of XGBoost over sklearn's GradientBoosting?
    "q5": "",  # If your boosting model overfits, name two hyperparameters you would adjust
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Boosting"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
